# CAMPUS NAVIGATION & N-QUEENS
## AI Search Algorithms – Updated Jupyter Notebook

**Objective:** Implement and compare uninformed, informed, local search, and constraint-satisfaction techniques using a campus navigation problem and the N-Queens problem.

## 1. Import Required Libraries

In [ ]:
import time
import heapq
from collections import deque

## 2. Campus Navigation Graph

The graph represents campus locations. Edge weights represent approximate walking distances in metres.

In [ ]:
graph = {
    "Main Gate": {"Library": 120, "Canteen": 80, "Administrative Block": 100},
    "Library": {"Workshop": 100, "Computer Department": 140},
    "Canteen": {"Parking Area": 60, "Workshop": 90},
    "Administrative Block": {"Computer Department": 80},
    "Parking Area": {"Mechanical Department": 180},
    "Workshop": {"Mechanical Department": 70},
    "Computer Department": {"Mechanical Department": 120},
    "Mechanical Department": {}
}

## 3. Admissible Heuristic Values

The heuristic estimates the remaining distance to the Mechanical Department. These values are chosen so they do **not overestimate** the actual shortest remaining distance for this graph, which makes the heuristic admissible for A*.

In [ ]:
h = {
    "Main Gate": 200,
    "Library": 170,
    "Canteen": 160,
    "Administrative Block": 200,
    "Parking Area": 180,
    "Workshop": 70,
    "Computer Department": 120,
    "Mechanical Department": 0
}

## 4. Breadth First Search (BFS)

**Type:** Uninformed Search

BFS explores nodes level by level using a queue. On this weighted graph, it minimizes the number of edges rather than total distance.

In [ ]:
def bfs(start, goal):
    queue = deque([(start, [start], 0)])
    visited = set()
    nodes = 0
    while queue:
        current, path, cost = queue.popleft()
        nodes += 1
        if current == goal:
            return path, cost, nodes
        if current in visited:
            continue
        visited.add(current)
        for location, distance in graph[current].items():
            if location not in visited:
                queue.append((location, path + [location], cost + distance))
    return None, 0, nodes

## 5. Depth First Search (DFS)

**Type:** Uninformed Search

DFS explores deeply before backtracking, using a stack.

In [ ]:
def dfs(start, goal):
    stack = [(start, [start], 0)]
    visited = set()
    nodes = 0
    while stack:
        current, path, cost = stack.pop()
        nodes += 1
        if current == goal:
            return path, cost, nodes
        if current in visited:
            continue
        visited.add(current)
        neighbours = list(graph[current].items())
        neighbours.reverse()
        for location, distance in neighbours:
            if location not in visited:
                stack.append((location, path + [location], cost + distance))
    return None, 0, nodes

## 6. Greedy Best-First Search

**Type:** Informed Search

Greedy search chooses the next location with the smallest heuristic value. It does not guarantee an optimal weighted path.

In [ ]:
def greedy(start, goal):
    current = start
    path = [current]
    cost = 0
    visited = set()
    nodes = 0
    while current != goal:
        visited.add(current)
        nodes += 1
        candidates = [location for location in graph[current] if location not in visited]
        if not candidates:
            return None, 0, nodes
        next_location = min(candidates, key=lambda location: h[location])
        cost += graph[current][next_location]
        current = next_location
        path.append(current)
        nodes += 1
    return path, cost, nodes

## 7. A* Search

**Type:** Informed Search

A* evaluates nodes using:

$$f(n) = g(n) + h(n)$$

With the admissible heuristic above, A* can guarantee an optimal path for this graph.

In [ ]:
def a_star(start, goal):
    priority_queue = [(h[start], 0, start, [start])]
    best_cost = {start: 0}
    nodes = 0
    while priority_queue:
        f, g, current, path = heapq.heappop(priority_queue)
        if g != best_cost.get(current):
            continue
        nodes += 1
        if current == goal:
            return path, g, nodes
        for location, distance in graph[current].items():
            new_cost = g + distance
            if location not in best_cost or new_cost < best_cost[location]:
                best_cost[location] = new_cost
                new_f = new_cost + h[location]
                heapq.heappush(priority_queue, (new_f, new_cost, location, path + [location]))
    return None, 0, nodes

## 8. Hill Climbing

**Type:** Local Search

Hill climbing repeatedly moves to a neighbouring state with a better heuristic value. It can get stuck at a local optimum.

In [ ]:
def hill_climbing(start, goal):
    current = start
    path = [current]
    cost = 0
    nodes = 0
    while current != goal:
        nodes += 1
        neighbours = list(graph[current].items())
        if not neighbours:
            return None, 0, nodes
        next_location, distance = min(neighbours, key=lambda item: h[item[0]])
        if h[next_location] >= h[current]:
            return None, cost, nodes
        cost += distance
        current = next_location
        path.append(current)
        nodes += 1
    return path, cost, nodes

## 9. N-Queens Problem

**Type:** Constraint Satisfaction Problem (CSP) using Backtracking

The goal is to place N queens on an N × N board so that no two queens share a column or diagonal.

In [ ]:
def solve_n_queens(n):
    board = [-1] * n
    nodes = 0
    def is_safe(row, col):
        for previous_row in range(row):
            previous_col = board[previous_row]
            if previous_col == col:
                return False
            if abs(previous_col - col) == abs(previous_row - row):
                return False
        return True
    def backtrack(row):
        nonlocal nodes
        nodes += 1
        if row == n:
            return True
        for col in range(n):
            if is_safe(row, col):
                board[row] = col
                if backtrack(row + 1):
                    return True
                board[row] = -1
        return False
    start_time = time.perf_counter()
    solved = backtrack(0)
    end_time = time.perf_counter()
    return solved, board, nodes, end_time - start_time

## 10. Performance Testing

In [ ]:
def test_algorithm(name, function):
    start_time = time.perf_counter()
    path, cost, nodes = function("Main Gate", "Mechanical Department")
    end_time = time.perf_counter()
    execution_time = end_time - start_time
    result = {
        "Algorithm": name,
        "Path": " -> ".join(path) if path else "No solution",
        "Cost (m)": cost if path else None,
        "Nodes Explored": nodes,
        "Execution Time (s)": execution_time
    }
    print("\n" + "-" * 70)
    print("Algorithm:", name)
    if path:
        print("Path:", " -> ".join(path))
        print("Total Cost:", cost, "m")
    else:
        print("No solution found")
    print("Nodes Explored:", nodes)
    print("Execution Time:", execution_time, "seconds")
    return result

## 11. Run All Search Algorithms

In [ ]:
print("=" * 70)
print("CAMPUS NAVIGATION - SEARCH ALGORITHM PERFORMANCE EVALUATION")
print("=" * 70)
algorithms = [
    ("Breadth First Search", bfs),
    ("Depth First Search", dfs),
    ("Greedy Best-First Search", greedy),
    ("A* Search", a_star),
    ("Hill Climbing", hill_climbing)
]
results = [test_algorithm(name, function) for name, function in algorithms]

## 12. Search Algorithm Comparison

In [ ]:
import pandas as pd
results_df = pd.DataFrame(results)
display(results_df)

## 13. N-Queens Test

In [ ]:
print("\n" + "=" * 70)
print("CONSTRAINT SATISFACTION - N QUEENS")
print("=" * 70)
n = 4
solved, board, nodes, execution_time = solve_n_queens(n)
if solved:
    print("N =", n)
    print("Solution:", board)
    print("Nodes Explored:", nodes)
    print("Execution Time:", execution_time, "seconds")
else:
    print("No solution found")

## 14. Visualise the N-Queens Solution

In [ ]:
if solved:
    print("\nN-Queens Board:\n")
    for col in board:
        row = ["."] * n
        row[col] = "Q"
        print(" ".join(row))

## Conclusion

- **BFS** minimizes the number of edges, not weighted distance.
- **DFS** does not guarantee the shortest weighted path.
- **Greedy Best-First Search** uses only the heuristic and is not guaranteed to be optimal.
- **A\*** combines path cost and heuristic information. With the corrected admissible heuristic, it returns the optimal 240 m route for this graph.
- **Hill Climbing** is a local search technique and can get stuck at local optima.
- **N-Queens** demonstrates constraint satisfaction through recursive backtracking.